# GENERAL INDEX PYTHON SCREENING TASK

#### Import basic libraries

In [3]:
import pandas as pd
from datetime import datetime
import pytz

#### Load CSV + Basic EDA

In [5]:
#Load the csv data
df = pd.read_csv("source.csv")

In [6]:
print(df.head())
print("Shape of the dataset:", df.shape)

       Name                 Datetime  Amount  Price  Purity
0  ProductA  2022-01-01T01:00:00.000      10  22.09  Impure
1  ProductA  2022-01-01T02:00:00.000      15  24.22    Pure
2  ProductA  2022-01-01T03:00:00.000      10  25.96  Impure
3  ProductA  2022-01-01T04:00:00.000      20  21.16  Impure
4  ProductA  2022-01-01T05:00:00.000      10  20.05    Pure
Shape of the dataset: (24, 5)


In [7]:
# Check for missing values
print(df.isnull().sum())

Name        0
Datetime    0
Amount      0
Price       0
Purity      0
dtype: int64


#### Convert datetime to UTC+6

In [9]:
#Convert datetime column from UTC to UTC+6
df["Datetime"] = pd.to_datetime(df["Datetime"], utc=True)
df["Datetime"] = df["Datetime"].dt.tz_convert("Etc/GMT-6")

#### Calculate total

In [11]:
# Split ProductA and ProductB
product_a = df[df['Name'] == 'ProductA'][['Datetime','Price','Purity']]
product_a = product_a.rename(columns={'Price':'A_Price','Purity':'A_Purity'})

# Merge back to get Product A price info for Product B
df = df.merge(product_a, on='Datetime', how='left')

# Calculate total
def calc_total(row):
    price = row['Price'] * 0.75 if row['Purity']=='Impure' else row['Price']
    
    if row['Name']=='ProductA':
        return row['Amount'] * price
    else:  # ProductB
        a_price = row['A_Price'] * 0.75 if row['A_Purity']=='Impure' else row['A_Price']
        return row['Amount'] * (price - a_price)

df['Total'] = df.apply(calc_total, axis=1)

In [12]:
# Preview results
df.head(10)

,Name,Datetime,Amount,Price,Purity,A_Price,A_Purity,Total
0,ProductA,2022-01-01 07:00:00+06:00,10,22.09,Impure,22.09,Impure,165.6750
1,ProductA,2022-01-01 08:00:00+06:00,15,24.22,Pure,24.22,Pure,363.3000
2,ProductA,2022-01-01 09:00:00+06:00,10,25.96,Impure,25.96,Impure,194.7000
3,ProductA,2022-01-01 10:00:00+06:00,20,21.16,Impure,21.16,Impure,317.4000
4,ProductA,2022-01-01 11:00:00+06:00,10,20.05,Pure,20.05,Pure,200.5000
5,ProductA,2022-01-01 12:00:00+06:00,10,21.27,Impure,21.27,Impure,159.5250
6,ProductA,2022-01-01 13:00:00+06:00,20,20.08,Pure,20.08,Pure,401.6000
7,ProductA,2022-01-01 14:00:00+06:00,10,22.19,Pure,22.19,Pure,221.9000
8,ProductA,2022-01-01 15:00:00+06:00,15,21.23,Impure,21.23,Impure,238.8375
9,ProductA,2022-01-01 16:00:00+06:00,20,22.34,Pure,22.34,Pure,446.8000


#### Save results

In [14]:
# Drop the helper columns
df = df.drop(columns=['A_Price', 'A_Purity'])

# Save to result.csv
df.to_csv('result.csv', index=False)